# Vendor Products Data Analysis

**Scenario:** A cosmetics and perfume store sells products online. Their ERP system exports the catalog as an Excel file (`VendorProducts.xlsx`) with **two sheets**:

| Sheet | What it holds | Rows |
|-------|---------------|------|
| `Items` | Every sellable product variant | 24,702 |
| `Collections` | Parent products and their color/size variants | 14,486 |

The two sheets are messy and inconsistent:

- Prices are text like `"22,500د.ع"` (Iraqi Dinar) — not numbers
- `Collections` mixes the color/size into the name and is missing columns that `Items` has
- Column names differ between sheets

**Goal:** clean both sheets, make them match, and stack them into **one unified product catalog**.

**What you will learn:**

- Loading multiple Excel sheets with pandas
- Cleaning messy text columns with the vectorized `.str` accessor
- Transforming (combining, renaming, dropping) columns
- Enriching one table with data from another using `pd.merge()`
- Stacking tables with `pd.concat()` and exporting the result

## Setup

Import pandas — the only library needed.

In [ ]:
import pandas as pd

## Load Data

Read both sheets from the Excel file.

In [ ]:
items = pd.read_excel('VendorProducts.xlsx', sheet_name='Items')
cols = pd.read_excel('VendorProducts.xlsx', sheet_name='Collections')

### Quick Look at the Raw Data

Before touching the data, check the shape and how many **unique products** each sheet holds. Note that `Items` can list the same product `Id` multiple times — one row per variant (size / color).

In [ ]:
print("Items sheet      ->", items.shape)
print("Collections sheet ->", cols.shape)
print()
print("Unique product IDs in Items      ->", items['Id'].nunique())
print("Unique product IDs in Collections ->", cols['Item Id'].nunique())

## Step 1: Clean Prices

Prices are stored as text with a currency suffix (`د.ع`, Iraqi Dinar) and thousand-separators (`"22,500"`). We strip both, then convert to integers so we can sort, group, and compute statistics.

In [ ]:
# Clean prices on BOTH sheets: strip the currency suffix and thousands separator,
# then convert the text to integers.
# pd.to_numeric(errors="coerce") turns anything unparseable into NaN, which we fill with 0
# so the pipeline never crashes on a bad cell.
items['Price'] = pd.to_numeric(items['Price'].str.replace('د.ع', '').str.replace(',', ''), errors='coerce').fillna(0).astype(int)
cols['Price'] = pd.to_numeric(cols['Price'].str.replace('د.ع', '').str.replace(',', ''), errors='coerce').fillna(0).astype(int)

print("Price range in Items (min -> max):", items['Price'].min(), "->", items['Price'].max())

> **Data-quality red flag:** the cheapest item has a price of `0`. A free product is almost certainly a data-entry error — real-world analysis always involves **deciding how to handle bad data** before trusting the numbers.

## Step 2: Transform Collections Sheet

### 2a. Merge Name + Color

Append the Color (Arabic) to Item Name and Secondary Color (English) to Secondary Item Name. This gives each variant a unique product name that includes its color/size.

`fillna('')` handles products that have no color value. `str.strip()` removes extra spaces.

In [ ]:
cols['Item Name'] = cols['Item Name'] + ' ' + cols['Color'].fillna('')
cols['Secondary Item Name'] = cols['Secondary Item Name'] + ' ' + cols['Secondary Color'].fillna('')
cols['Item Name'] = cols['Item Name'].str.strip()
cols['Secondary Item Name'] = cols['Secondary Item Name'].str.strip()

### 2b. Drop Unwanted Columns

Drop columns that are not in the final output: Color, Secondary Color, Size, Code, and Color Code.

In [ ]:
cols.drop(columns=['Color', 'Secondary Color', 'Size', 'Code', 'Color Code'], inplace=True)

### 2c. Rename Columns to Match Items Sheet

Rename so column names are identical between both sheets (needed for concat later).

In [ ]:
cols.rename(columns={
    'Item Id': 'Id',
    'Item Name': 'Name',
    'Secondary Item Name': 'Secondary Name'
}, inplace=True)

After renaming, Collections has these columns:
- `Id`, `Name`, `Secondary Name`, `Price`, `Barcode`, `Unit Level`, `Is Active`, `Picture`

## Step 3: Clean Items Sheet

Drop columns we don't need: Code, Sub Description, and Secondary Sub Description. Price was already cleaned in Step 1.

In [ ]:
items.drop(columns=['Code', 'Sub Description', 'Secondary Sub Description'], inplace=True)

After dropping, Items has these 12 columns:
- `Picture`, `Id`, `Name`, `Secondary Name`, `Barcode`, `Price`, `Description`, `Secondary Description`, `Menu`, `Brand`, `Secondary Brand`, `Has Collections`

## Step 4: Fill Missing Columns in Collections

Collections is missing these columns from Items:
- `Description`, `Secondary Description`, `Menu`, `Brand`, `Secondary Brand`, `Has Collections`

We map them from the Items sheet by matching on `Id` (every product in Collections also exists in Items).

In [ ]:
lookup = items[['Id', 'Description', 'Secondary Description', 'Menu', 'Brand', 'Secondary Brand', 'Has Collections']]
cols = pd.merge(cols, lookup, on='Id', how='left')

## Step 5: Drop Collection-Only Columns

Drop `Unit Level` and `Is Active` — these only exist in Collections and are not in the final Items column structure. Both sheets need the same columns for stacking.

In [ ]:
cols.drop(columns=['Unit Level', 'Is Active'], inplace=True)

Now both sheets have the same 12 columns.

**Items:** `Picture`, `Id`, `Name`, `Secondary Name`, `Barcode`, `Price`, `Description`, `Secondary Description`, `Menu`, `Brand`, `Secondary Brand`, `Has Collections`

**Collections:** `Id`, `Name`, `Secondary Name`, `Price`, `Barcode`, `Picture`, `Description`, `Secondary Description`, `Menu`, `Brand`, `Secondary Brand`, `Has Collections`

## Step 6: Stack Both Sheets

Concatenate Items (24,702 rows) and Collections (14,486 rows) into one unified dataframe. `ignore_index=True` resets the row numbers.

In [ ]:
final = pd.concat([items, cols], ignore_index=True)

## Step 7: Reorder Columns

Arrange columns to match the original Items sheet order.

In [ ]:
final = final[['Picture', 'Id', 'Name', 'Secondary Name', 'Barcode', 'Price',
               'Description', 'Secondary Description', 'Menu', 'Brand',
               'Secondary Brand', 'Has Collections']]

## Step 8: Verify and Export

Check the final shape and a preview before saving.

In [ ]:
print('Final shape:', final.shape)
final.tail(10)

### What Did We Learn From the Data?

The unified catalog now has **39,188 rows** (24,702 items + 14,486 collections) and a consistent 12-column structure. A few things the cleaned data lets us answer instantly:

- The most expensive products are luxury perfumes (e.g. Chanel Bleu at 300,000 د.ع)
- Most products share the same standard price point (22,500 د.ع)
- Both Arabic and English names are kept — useful for a bilingual storefront
- Every collection is linked back to its parent item through `Id`

In [ ]:
final.to_excel('AnalysedVendorProducts.xlsx', index=False)

## Done ✅

The output file **`AnalysedVendorProducts.xlsx`** now contains all Items rows followed by all Collections rows (stacked vertically) with a matching 12-column structure.

Run the final cell again after changing any of the cleaning steps to regenerate the file.

## 🎯 Key Takeaways

- `pd.read_excel(..., sheet_name=...)` loads a specific sheet — call it once per sheet.
- Messy text columns can be cleaned in one vectorized line with `.str.replace()` + `pd.to_numeric(errors="coerce")`.
- `pd.merge(df1, df2, on='Id', how='left')` enriches a table with columns from another (like a SQL `LEFT JOIN`).
- `pd.rename(columns={...})` aligns column names so tables can be stacked.
- `pd.concat([a, b], ignore_index=True)` stacks rows and resets the index.
- Always verify shape and previews before exporting — garbage in, garbage out.

## 🏋️ Practice Exercises

1. Find the **10 most expensive products** in the final catalog (`sort_values` + `head`).
2. How many products cost more than 50,000 د.ع? Use a boolean filter and `.shape[0]`.
3. Which **brand** has the most products? (`value_counts().head()`).
4. Group the final catalog by `Brand` and compute the average price per brand.
5. Export only the Arabic columns (`Name`, `Description`, `Menu`, `Brand`) to a CSV called `arabic_catalog.csv`.

## 🚀 Next Steps

- Review **`01_Pandas_Fundamentals.ipynb`** for the core pandas operations used here (filtering, groupby, merge, concat).
- Move on to **`04_File_Handling_and_IO.ipynb`** to learn text-file I/O.
- Then visualize this catalog with **`09_Matplotlib_Visualization.ipynb`** — e.g. a bar chart of the most common brands.